In [ ]:
pip install customtkinter

In [2]:
pip install pillow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import customtkinter as ctk
from tkinter import messagebox
from PIL import Image, ImageTk
import mysql.connector as ms


conn = ms.connect(
    host="localhost",
    user="root",
    password="root",
    database="login"
)
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS users (
        username VARCHAR(255) PRIMARY KEY,
        password VARCHAR(255)
    )
""")
conn.commit()


ctk.set_appearance_mode("light")
ctk.set_default_color_theme("blue")


def toggle_mode(choice):
    ctk.set_appearance_mode(choice)

def toggle_password():
    if password_entry.cget("show") == "":
        password_entry.configure(show="*")
        eye_btn.configure(text="👁")
    else:
        password_entry.configure(show="")
        eye_btn.configure(text="🙈")

def login():
    user = username_entry.get()
    pwd = password_entry.get()
    cursor.execute("SELECT * FROM users WHERE username=%s AND password=%s", (user, pwd))
    if cursor.fetchone():
        messagebox.showinfo("Login Success", f"Welcome, {user}!")
    else:
        messagebox.showerror("Login Failed", "Invalid username or password.")

def register_window():
    def register():
        user = new_username.get()
        pwd = new_password.get()
        try:
            cursor.execute("INSERT INTO users (username, password) VALUES (%s, %s)", (user, pwd))
            conn.commit()
            messagebox.showinfo("Success", "User registered successfully.")
            register_win.destroy()
        except ms.IntegrityError:
            messagebox.showerror("Error", "Username already exists.")

    register_win = ctk.CTkToplevel()
    register_win.title("Register")
    register_win.geometry("350x220")

    ctk.CTkLabel(register_win, text="New Username:").pack(pady=5)
    new_username = ctk.CTkEntry(register_win)
    new_username.pack(pady=5)

    ctk.CTkLabel(register_win, text="New Password:").pack(pady=5)
    new_password = ctk.CTkEntry(register_win, show="*")
    new_password.pack(pady=5)

    ctk.CTkButton(register_win, text="Register", command=register).pack(pady=15)

def forgot_password():
    def recover():
        user = forgot_user.get()
        cursor.execute("SELECT * FROM users WHERE username=%s", (user,))
        result = cursor.fetchone()
        if result:
            messagebox.showinfo("Note", "Passwords are securely stored and cannot be retrieved. Please contact admin.")
        else:
            messagebox.showerror("Error", "Username not found.")
        forgot_win.destroy()

    forgot_win = ctk.CTkToplevel()
    forgot_win.title("Forgot Password")
    forgot_win.geometry("300x160")

    ctk.CTkLabel(forgot_win, text="Enter your username:").pack(pady=5)
    forgot_user = ctk.CTkEntry(forgot_win)
    forgot_user.pack(pady=5)

    ctk.CTkButton(forgot_win, text="Recover Info", command=recover).pack(pady=10)


root = ctk.CTk()
root.title("Secure Login System")
root.geometry("1024x600")




frame = ctk.CTkFrame(master=root, width=420, height=520, corner_radius=15)
frame.place(relx=0.5, rely=0.5, anchor="center")


try:
    login_icon = ctk.CTkImage(light_image=Image.open("icon.png"), size=(80, 80))
    ctk.CTkLabel(frame, image=login_icon, text="").pack(pady=(20, 5))
except:
    pass  # In case image is missing


ctk.CTkLabel(frame, text="LOGIN SYSTEM", font=("Helvetica", 24, "bold")).pack(pady=10)


ctk.CTkLabel(frame, text="Username:", font=("Helvetica", 14)).pack(pady=(10, 2))
username_entry = ctk.CTkEntry(frame, font=("Helvetica", 12), width=250)
username_entry.pack(pady=5)


ctk.CTkLabel(frame, text="Password:", font=("Helvetica", 14)).pack(pady=(10, 2))
password_frame = ctk.CTkFrame(frame, fg_color="transparent")
password_frame.pack()

password_entry = ctk.CTkEntry(password_frame, show="*", width=210)
password_entry.pack(side="left", padx=(0, 10), pady=5)

eye_btn = ctk.CTkButton(password_frame, text="👁", width=40, command=toggle_password)
eye_btn.pack(side="left", pady=5)


ctk.CTkButton(frame, text="Login", command=login, width=200).pack(pady=15)
ctk.CTkButton(frame, text="Register", command=register_window, width=200).pack(pady=5)
ctk.CTkButton(frame, text="Forgot Password", command=forgot_password, width=200).pack(pady=5)


appearance = ctk.CTkOptionMenu(root, values=["Light", "Dark"], command=toggle_mode)
appearance.set("Light")
appearance.place(x=20, y=20)

ctk.CTkButton(root, text="Exit", command=root.destroy, fg_color="red", hover_color="#990000", width=80).place(x=20, y=70)

root.mainloop()
